<a href="https://colab.research.google.com/github/AliyaRanawat/Social-Media-Forensics-LLM/blob/main/REAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q datasets sentence-transformers gradio

In [ ]:
import gc
import warnings
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
import gradio as gr

warnings.filterwarnings("ignore")

print("INITIALIZING 1-LAKH GEN-Z & IMPLICIT TOXICITY PIPELINE...")

# Check for GPU Acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Processing Device: {device.upper()}")

# ==========================================
# 1. 1-LAKH DATASET INGESTION & GEN-Z BALANCING
# ==========================================
print("\nIngesting & Building 1,00,000 Sample Dataset...")

try:
    dataset = load_dataset(
        "thesofakillers/jigsaw-toxic-comment-classification-challenge",
        split="train",
    )
    df = dataset.to_pandas()

    toxicity_cols = [
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate",
    ]
    df["label"] = (
        df[toxicity_cols].sum(axis=1).apply(lambda x: 1 if x > 0 else 0)
    )

    # 47,500 Explicit Toxic + 47,500 Safe = 95,000 Baseline Samples
    toxic_df = df[df["label"] == 1].sample(
        n=47500, random_state=42, replace=True
    )
    safe_df = df[df["label"] == 0].sample(n=47500, random_state=42)

    df_massive = (
        pd.concat([toxic_df, safe_df])[["comment_text", "label"]]
        .rename(columns={"comment_text": "text"})
    )

except Exception as e:
    print(f"Hugging Face load failed, using fallback. Error: {e}")
    df_massive = pd.DataFrame(columns=["text", "label"])

# GEN Z SARCASTIC ROASTS & PASSIVE-AGGRESSIVE IMPLICIT TOXICITY (TOXIC = 1)
genz_toxic_data = [
    ("Bro thought he cooked", 1),
    ("Who let bro cook", 1),
    ("Main character syndrome is crazy with this one", 1),
    ("It's the confidence for me", 1),
    ("Bro really thought we would agree", 1),
    ("Not you thinking you did something here", 1),
    ("Side eye... bombastic side eye", 1),
    ("Bro is yapping about absolute nothing", 1),
    ("Giving desperate honestly", 1),
    ("A for effort, F for outcome", 1),
    ("The math isn't mathing with your brain", 1),
    ("Pick me energy is strong here", 1),
    ("Bro is talking to the wall", 1),
    ("Who asked though?", 1),
    ("Nice opinion, did your mom give it to you?", 1),
    ("Bro turned off his brain before posting", 1),
    ("Imagine being this confident while being so wrong", 1),
    ("You're doing great sweetie... said no one ever", 1),
    ("I envy people who haven’t met you yet", 1),
    ("you're the reason the gene pool needs a lifeguard", 1),
    ("what a courage to wear such beautiful dress on that body", 1),
    ("did you bath today?", 1),
    ("your voice sounds so annoying in this video", 1),
    ("It's so brave of you to wear that in public.", 1),
    ("You're actually pretty smart for someone who looks like you.", 1),
    ("I love how you just don't care about your appearance.", 1),
    ("You did surprisingly well considering your background.", 1),
    ("You look so much better when you actually try.", 1),
    ("bless your heart, you tried", 1),
    ("you are so articulate for someone from your neighborhood", 1),
    ("That is certainly an interesting choice of clothes.", 1),
    ("Wow Einstein, you are a literal genius for doing that.", 1),
    ("I hope your day is as pleasant as you are.", 1),
    ("You bring so much joy when you leave the room.", 1),
    ("You're pretty for your age.", 1),
    ("I wish I had your confidence to post a picture like that.", 1),
    ("You're remarkably bright considering your school.", 1),
    ("You look so healthy now, you used to be so skinny.", 1),
    ("It's amazing how you don't care what people think of you.", 1),
    ("You actually cleaned up nice today.", 1),
    ("Must be nice to have so much free time to do nothing.", 1),
    ("You sound so intelligent when you don't speak.", 1),
    ("You're full of surprises today, usually you mess everything up.", 1),
    ("I love how you can wear anything and not care how ridiculous it looks.", 1),
    ("You did great, I didn't expect much from you anyway.", 1),
    ("You are living proof that mistakes happen.", 1),
    ("I'd agree with you but then we'd both be wrong.", 1),
    ("You have an interesting face.", 1),
    ("Bro is typing with his feet", 1),
    ("Delete this while you still have time", 1)
]

# GEN Z WHOLESOME SLANG & POSITIVE COMMENTS (SAFE = 0)
genz_safe_data = [
    ("Bro actually cooked with this video", 0),
    ("This is actually so valid", 0),
    ("You ate and left no crumbs", 0),
    ("Main character energy in the best way!", 0),
    ("W post, keep it up brother", 0),
    ("No cap this is so wholesome", 0),
    ("Bro dropped an absolute banger", 0),
    ("Slay queen, killed it!", 0),
    ("Understood the assignment 100%", 0),
    ("Real for this, total respect", 0),
    ("This made my whole day honestly", 0),
    ("You are literally so talented", 0),
    ("Pure perfection, no notes", 0),
    ("The vibe here is immaculate", 0),
    ("Sending you so much love and support", 0),
    ("You dropped this", 0),
    ("Honestly so proud of you", 0),
    ("This is the best thing I've seen all week", 0),
    ("Keep shining, don't let anyone stop you", 0),
    ("10/10 content right here", 0),
    ("This deserves way more views", 0),
    ("Unmatched energy, love it!", 0),
    ("So inspiring, keep going!", 0),
    ("You have such a kind soul", 0),
    ("Brilliant idea, love how you executed it", 0),
    ("Such wholesome content", 0),
    ("This is cinema", 0),
    ("Absolute masterpiece", 0),
    ("Thank you for sharing this!", 0),
    ("Never stop making these videos!", 0)
]

# Create DataFrames
df_genz_toxic = pd.DataFrame(genz_toxic_data, columns=["text", "label"])
df_genz_safe = pd.DataFrame(genz_safe_data, columns=["text", "label"])

# OVERSAMPLING CUSTOM DATA TO BUILD 5,000 SAMPLES:
# 50 toxic examples * 70 = 3,500 samples
# 30 safe examples * 50 = 1,500 samples
# Total Custom = 5,000 samples
df_toxic_upsampled = pd.concat([df_genz_toxic] * 70, ignore_index=True)
df_safe_upsampled = pd.concat([df_genz_safe] * 50, ignore_index=True)

# COMBINE ALL DATASETS: 95,000 (Jigsaw) + 5,000 (Gen-Z & Implicit) = EXACT 1,00,000
df_final = pd.concat(
    [df_massive, df_toxic_upsampled, df_safe_upsampled], ignore_index=True
)
df_final["text"] = df_final["text"].astype(str).str.replace("\n", " ")
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Train/Test Split (80/20)
train_size = int(0.8 * len(df_final))
df_tr = df_final.iloc[:train_size]
df_te = df_final.iloc[train_size:]

print(
    f"1-Lakh Dataset Ready! Total Samples: {len(df_final)} | Train: {len(df_tr)} | Test: {len(df_te)}"
)

# Free raw DataFrames from memory
del df, df_massive, toxic_df, safe_df, df_genz_toxic, df_genz_safe, df_toxic_upsampled, df_safe_upsampled
gc.collect()

# ==========================================
# 2. GPU EMBEDDING EXTRACTION
# ==========================================
print("\nLoading Contextual Transformer (all-MiniLM-L12-v2)...")
llm = SentenceTransformer("all-MiniLM-L12-v2", device=device)

def get_embeddings_in_batches(texts, batch_size=2048):
    embeddings = []
    total_batches = (len(texts) // batch_size) + 1
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        emb = llm.encode(
            batch_texts,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        embeddings.append(emb)
        if (i // batch_size) % 5 == 0 or (i // batch_size) == total_batches - 1:
            print(f"   -> Processed batch {i // batch_size + 1}/{total_batches}...")
    return np.vstack(embeddings)

print("Extracting Training Embeddings...")
X_tr = get_embeddings_in_batches(df_tr["text"].tolist())
y_tr = df_tr["label"].values

print("Extracting Testing Embeddings...")
X_te = get_embeddings_in_batches(df_te["text"].tolist())
y_te = df_te["label"].values

del df_tr, df_te, df_final
gc.collect()

# ==========================================
# 3. ENSEMBLE CLASSIFIER (OPTIMIZED FOR SARCASM)
# ==========================================
print("\nTraining Voting Ensemble...")

m_hgb = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.08, l2_regularization=1.0, random_state=42
)

m_et = ExtraTreesClassifier(
    n_estimators=100, max_depth=25, random_state=42, n_jobs=-1
)

m_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=250,
    early_stopping=True,
    alpha=0.01,
    random_state=42,
)

my_est = [("HistGradientBoost", m_hgb), ("ExtraTrees", m_et), ("MLP_NN", m_mlp)]

voting_clf = VotingClassifier(estimators=my_est, voting="soft", n_jobs=-1)
voting_clf.fit(X_tr, y_tr)

# ==========================================
# 4. EVALUATION
# ==========================================
print("\nEvaluating Model Performance...")
preds = voting_clf.predict(X_te)
acc = accuracy_score(y_te, preds)

print("\n" + "=" * 50)
print(f"VALIDATED TEST ACCURACY: {acc * 100:.2f}%")
print("=" * 50)
print("\nDetailed Classification Report:")
print(classification_report(y_te, preds, target_names=["SAFE", "TOXIC"]))

# ==========================================
# 5. INTERACTIVE GRADIO UI
# ==========================================
print("\nLaunching Web Interface...")

def analyze_comment(comment):
    if not comment.strip():
        return "Empty", "0.00%"

    emb = llm.encode(
        [comment.strip()], show_progress_bar=False, normalize_embeddings=True
    )
    pred = voting_clf.predict(emb)[0]
    prob = voting_clf.predict_proba(emb)[0]
    conf = prob[pred] * 100

    if pred == 1:
        return "TOXIC / CYBERBULLYING", f"{conf:.2f}%"
    else:
        return "SAFE / WHOLESOME", f"{conf:.2f}%"

demo = gr.Interface(
    fn=analyze_comment,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type a social media comment or Gen Z slang here...",
        label="Enter Comment",
    ),
    outputs=[
        gr.Label(label="Prediction"),
        gr.Text(label="Confidence Score"),
    ],
    title="Social Media Forensics: Adaptive Cyberbullying Detection",
    description="Trained on 1,00,000 samples including explicit toxicity, Gen Z slang, passive-aggressiveness, and wholesome praise.",
    examples=[
        ["Bro thought he cooked"],
        ["Bro actually cooked with this video"],
        ["You ate and left no crumbs"],
        ["It's the confidence for me"],
        ["I envy people who haven’t met you yet"],
        ["what a courage to wear such beautiful dress on that body"],
        ["You dropped this"],
        ["Have a wonderful day!"],
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True, debug=False)

INITIALIZING 1-LAKH GEN-Z & IMPLICIT TOXICITY PIPELINE...
Processing Device: CUDA

Ingesting & Building 1,00,000 Sample Dataset...
1-Lakh Dataset Ready! Total Samples: 100000 | Train: 80000 | Test: 20000

Loading Contextual Transformer (all-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Extracting Training Embeddings...
   -> Processed batch 1/40...
   -> Processed batch 6/40...
   -> Processed batch 11/40...
   -> Processed batch 16/40...
   -> Processed batch 21/40...
   -> Processed batch 26/40...
   -> Processed batch 31/40...
   -> Processed batch 36/40...
   -> Processed batch 40/40...
Extracting Testing Embeddings...
   -> Processed batch 1/10...
   -> Processed batch 6/10...
   -> Processed batch 10/10...

Training Voting Ensemble...
